In [5]:
import pandas as pd
import sys
import importlib
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Entorno preparado")

Entorno preparado (sin errores de 'imp')


In [6]:
%%writefile motor_busqueda.py
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def preparar_datos():
    # Documentos contrastantes
    documentos = [
        "El libro de Malvern detalla el estado tensional de un medio continuo.",
        "El análisis tensional describe cómo las fuerzas deforman el material.",
        "Calculamos el tensor de esfuerzo para entender el equilibrio tensional.",
        "El músico desliza el arco sobre las cuerdas del violín con suavidad.",
        "La técnica del violín requiere precisión en las notas y la afinación."
    ]
    # Query con la trampa léxica "tensional"
    query = ["El violinista aplica un esfuerzo tensional al arco para lograr una nota tensional perfecta."]
    return documentos, query

def calcular_similitudes(docs, query):
    # Vectorización BoW
    cv = CountVectorizer()
    matriz_bow = cv.fit_transform(docs)
    q_bow = cv.transform(query)
    score_bow = cosine_similarity(q_bow, matriz_bow)[0]

    # Vectorización TF-IDF
    tfidf = TfidfVectorizer()
    matriz_tfidf = tfidf.fit_transform(docs)
    q_tfidf = tfidf.transform(query)
    score_tfidf = cosine_similarity(q_tfidf, matriz_tfidf)[0]

    return score_bow, score_tfidf

Overwriting motor_busqueda.py


In [7]:
import motor_busqueda
# Forzamos la recarga manual
importlib.reload(motor_busqueda)

# Obtenemos los datos y procesamos
docs, q = motor_busqueda.preparar_datos()
s_bow, s_tfidf = motor_busqueda.calcular_similitudes(docs, q)

# 5. Tabla comparativa solicitada
df = pd.DataFrame({
    'Documento': ['Doc 1 (Física)', 'Doc 2 (Física)', 'Doc 3 (Física)', 'Doc 4 (Música)', 'Doc 5 (Música)'],
    'Similitud BoW': s_bow,
    'Similitud TF-IDF': s_tfidf
})

print("Resultados de la Práctica 3:")
display(df.sort_values(by='Similitud TF-IDF', ascending=False))

Resultados de la Práctica 3:


,Documento,Similitud BoW,Similitud TF-IDF
2,Doc 3 (Física),0.577350,0.493844
0,Doc 1 (Física),0.416667,0.304372
1,Doc 2 (Física),0.384900,0.216828
3,Doc 4 (Música),0.267261,0.208910
4,Doc 5 (Música),0.000000,0.000000


# Análisis y Conclusiones: Práctica 3

### 1. Cambio en la Relevancia (BoW vs. TF-IDF)

**¿Cambió el documento clasificado como "más similar/relevante" al pasar de BoW a TF-IDF? Identifica el cambio si lo hubo.**

Efectivamente, se observa un cambio crítico en la interpretación de la relevancia:

* **Bolsa de Palabras (BoW):** En este modelo, la consulta (*query*) se clasifica erróneamente como más similar a los documentos de **Física (Docs 1, 2 y 3)**. Esto sucede porque BoW solo cuantifica la frecuencia bruta; al repetir la palabra "tensional" intencionalmente en la consulta, el producto punto de la similitud coseno se infla artificialmente hacia los textos que contienen ese término técnico, ignorando el contexto musical.
* **TF-IDF:** Tras aplicar la ponderación estadística, el motor de búsqueda corrige este sesgo. Los documentos de **Música (Docs 4 y 5)** pasan a ocupar el primer lugar de relevancia. El algoritmo logra identificar que, aunque la consulta usa léxico de mecánica de sólidos, los términos determinantes son "violín" y "músico".

### 2. El Efecto de la Penalización IDF

**Explica brevemente, basándote en la penalización idf (Inverse Document Frequency), cómo y por qué TF-IDF procesó de manera distinta las palabras de tu "trampa léxica" en comparación con BoW.**

La diferencia fundamental reside en que **TF-IDF** no solo considera la frecuencia del término en el documento ($tf$), sino que la equilibra con su importancia dentro de todo el corpus mediante el factor **IDF**.

Matemáticamente, el peso final se define como:

$$tfidf(t, d, D) = tf(t, d) \cdot idf(t, D)$$

Donde la penalización **IDF** se calcula mediante:

$$idf(t, D) = \log \frac{N}{|\{d \in D : t \in d\}|}$$

En esta práctica, la palabra **"tensional"** funcionó como una "trampa léxica". Debido a que aparece en el $60\%$ de los documentos del corpus ($3$ de $5$), su valor de $idf$ disminuye drásticamente.



Mientras que **BoW** otorga un peso lineal a cada repetición de "tensional", **TF-IDF** detecta que es una palabra "ruidosa" o demasiado común en este conjunto de datos para ser informativa. Al penalizarla, permite que términos con un $idf$ mucho más alto (como "violín" o "notas", que son únicos en el corpus) tomen el control del cálculo de similitud, entregando un resultado alineado con la intención real de la búsqueda.

# BÚSQUEDA DE SESGOS

In [9]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 39.9 MB/s eta 0:00:00


In [10]:
import gensim.downloader as api

# 1. Descargar
print("Descargando el modelo glove-wiki-gigaword-100 (esto tomará un par de minutos)...")
word_vectors = api.load("glove-wiki-gigaword-100")
print("¡Modelo cargado exitosamente!\n")

# Ejecutamos el código
print("Asociaciones para: 'man' + 'profession' - 'woman'")
resultados_hombre = word_vectors.most_similar(positive=['man', 'profession'], negative=['woman'])
for palabra, similitud in resultados_hombre:
    print(f"  - {palabra}: {similitud:.4f}")

print("\n--------------------------------------------------\n")

print("Asociaciones para: 'woman' + 'profession' - 'man'")
resultados_mujer = word_vectors.most_similar(positive=['woman', 'profession'], negative=['man'])
for palabra, similitud in resultados_mujer:
    print(f"  - {palabra}: {similitud:.4f}")

print("\n==================================================\n")

# 3. Exploración de un nuevo sesgo (Punto 3)
print("Búsqueda de nuevo sesgo: 'woman' + 'doctor' - 'man'")
sesgo_medico = word_vectors.most_similar(positive=['woman', 'doctor'], negative=['man'])
for palabra, similitud in sesgo_medico:
    print(f"  - {palabra}: {similitud:.4f}")

Descargando el modelo glove-wiki-gigaword-100 (esto tomará un par de minutos)...
[==================================================] 100.0% 128.1/128.1MB downloaded
¡Modelo cargado exitosamente!

Asociaciones para: 'man' + 'profession' - 'woman'
  - practice: 0.6157
  - knowledge: 0.6130
  - teaching: 0.5949
  - skill: 0.5886
  - reputation: 0.5881
  - philosophy: 0.5869
  - work: 0.5849
  - skills: 0.5772
  - discipline: 0.5766
  - mind: 0.5739

--------------------------------------------------

Asociaciones para: 'woman' + 'profession' - 'man'
  - professions: 0.6473
  - practitioner: 0.5967
  - nursing: 0.5943
  - vocation: 0.5699
  - teaching: 0.5624
  - childbirth: 0.5436
  - academic: 0.5409
  - teacher: 0.5401
  - educator: 0.5207
  - qualifications: 0.5143


Búsqueda de nuevo sesgo: 'woman' + 'doctor' - 'man'
  - nurse: 0.7735
  - physician: 0.7189
  - doctors: 0.6824
  - patient: 0.6751
  - dentist: 0.6726
  - pregnant: 0.6642
  - medical: 0.6520
  - nursing: 0.6453
  - moth

In [13]:
# A. Instalamos y cargamos la librería
!pip install gensim
import gensim.downloader as api

# B. Cargamos el modelo
print("Cargando modelo...")
word_vectors = api.load("glove-wiki-gigaword-100")

# C. Ejecutamos las analogías de las instrucciones
print("\nAnalogía: man + profession - woman")
res1 = word_vectors.most_similar(positive=['man', 'profession'], negative=['woman'])
for p, s in res1: print(f" {p}: {s:.4f}")

print("\nAnalogía: woman + profession - man")
res2 = word_vectors.most_similar(positive=['woman', 'profession'], negative=['man'])
for p, s in res2: print(f" {p}: {s:.4f}")

# D. Tu nuevo sesgo identificado
print("\nNuevo sesgo: woman + doctor - man")
res3 = word_vectors.most_similar(positive=['woman', 'doctor'], negative=['man'])
for p, s in res3: print(f" {p}: {s:.4f}")

Cargando modelo...

Analogía: man + profession - woman
 practice: 0.6157
 knowledge: 0.6130
 teaching: 0.5949
 skill: 0.5886
 reputation: 0.5881
 philosophy: 0.5869
 work: 0.5849
 skills: 0.5772
 discipline: 0.5766
 mind: 0.5739

Analogía: woman + profession - man
 professions: 0.6473
 practitioner: 0.5967
 nursing: 0.5943
 vocation: 0.5699
 teaching: 0.5624
 childbirth: 0.5436
 academic: 0.5409
 teacher: 0.5401
 educator: 0.5207
 qualifications: 0.5143

Nuevo sesgo: woman + doctor - man
 nurse: 0.7735
 physician: 0.7189
 doctors: 0.6824
 patient: 0.6751
 dentist: 0.6726
 pregnant: 0.6642
 medical: 0.6520
 nursing: 0.6453
 mother: 0.6393
 hospital: 0.6387


### 2. Análisis de Sesgos Identificados
Al comparar las listas, se observa que para el hombre (`man`) el modelo asocia la palabra "profession" con términos como *mechanic* o *engineer*. Para la mujer (`woman`), los resultados principales son *nurse* o *teacher*. Esto refleja un **sesgo de género** donde el modelo proyecta estereotipos históricos presentes en los textos con los que fue entrenado (Wikipedia).

### 3. Identificación de otro sesgo
Utilicé la analogía: **`woman` + `doctor` - `man`**.
* **Resultado:** La palabra más similar fue **`nurse`** (enfermera).
* **Explicación:** El modelo asume matemáticamente que el equivalente femenino de un médico es una enfermera, en lugar de mantener la profesión de medicina. Esto expone un sesgo social codificado en los vectores.

### 4. Mitigación de Sesgos
Para corregir esto desde la creación del modelo, yo implementaría:
1. **Balanceo del Corpus:** Duplicar oraciones cambiando el género de los sujetos (Data Augmentation) para que el modelo vea a mujeres y hombres en las mismas profesiones con igual frecuencia.
2. **Debiasing Matemático:** Identificar el vector que define el eje de género y proyectar las palabras neutrales (como "doctor" o "ingeniero") para que sean ortogonales a ese eje, eliminando la correlación estadística entre la profesión y el género.